In [0]:
from pyspark.sql.functions import current_timestamp

RAW_PATH = "abfss://data@stecommerceom2026.dfs.core.windows.net/raw/"

CATALOG = "dbw_ecommerce_om"
BRONZE_SCHEMA = "ecommerce"

print("Bronze ingestion configuration loaded")
print(f"Raw path: {RAW_PATH}")
print(f"Target: {CATALOG}.{BRONZE_SCHEMA}")

Bronze ingestion configuration loaded
Raw path: abfss://data@stecommerceom2026.dfs.core.windows.net/raw/
Target: dbw_ecommerce_om.ecommerce


In [0]:
customers_raw = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(RAW_PATH + "customers.csv")

products_raw = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(RAW_PATH + "products.csv")

orders_raw = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(RAW_PATH + "orders.csv")

order_items_raw = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(RAW_PATH + "order_items.csv")

payments_raw = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(RAW_PATH + "payments.csv")

print("All five raw CSV files loaded")

All five raw CSV files loaded


In [0]:
tables = {
    "bronze_customers": customers_raw,
    "bronze_products": products_raw,
    "bronze_orders": orders_raw,
    "bronze_order_items": order_items_raw,
    "bronze_payments": payments_raw
}

for table_name, df in tables.items():
    (
        df.withColumn("_ingestion_timestamp", current_timestamp())
          .write
          .format("delta")
          .mode("overwrite")
          .option("mergeSchema", "true")
          .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.{table_name}")
    )

    print(f"Written: {CATALOG}.{BRONZE_SCHEMA}.{table_name}")

print("Bronze ingestion completed successfully")

Written: dbw_ecommerce_om.ecommerce.bronze_customers
Written: dbw_ecommerce_om.ecommerce.bronze_products
Written: dbw_ecommerce_om.ecommerce.bronze_orders
Written: dbw_ecommerce_om.ecommerce.bronze_order_items
Written: dbw_ecommerce_om.ecommerce.bronze_payments
Bronze ingestion completed successfully


In [0]:
for table_name in tables:
    df = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.{table_name}")

    print(f"{table_name}: {df.count()} rows")
    print(f"Columns: {df.columns}")
    print("-" * 60)

print("Bronze validation completed")

bronze_customers: 10000 rows
Columns: ['customer_id', 'name', 'email', 'city', 'state', 'registration_date', '_ingestion_timestamp']
------------------------------------------------------------
bronze_products: 2000 rows
Columns: ['product_id', 'product_name', 'category', 'price', '_ingestion_timestamp']
------------------------------------------------------------
bronze_orders: 100100 rows
Columns: ['order_id', 'customer_id', 'order_date', 'status', 'total_amount', '_ingestion_timestamp']
------------------------------------------------------------
bronze_order_items: 200000 rows
Columns: ['order_item_id', 'order_id', 'product_id', 'quantity', 'price', '_ingestion_timestamp']
------------------------------------------------------------
bronze_payments: 100000 rows
Columns: ['payment_id', 'order_id', 'payment_method', 'payment_status', 'payment_date', '_ingestion_timestamp']
------------------------------------------------------------
Bronze validation completed


In [0]:
spark.table("dbw_ecommerce_om.ecommerce.bronze_orders").printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: double (nullable = true)
 |-- order_date: date (nullable = true)
 |-- status: string (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)

